# 02. Sentiment Analysis

---

## Business Objective
---

Generate a daily sentiment score from economic news to be used as a feature for market forecasting.

## 1. Data Preparation for Sentiment Analysis

In [1]:
import sys
import os

sys.path.append(
    os.path.abspath("..")
)

import pandas as pd

In [2]:
from src.data_io import (
load_dataframe,
save_dataframe
)

In [3]:
DATA_PATH_NEWS = "../data/raw/economy_news.parquet"
economy_df = load_dataframe(
    DATA_PATH_NEWS
)

economy_df.shape

(6867, 13)

In [4]:
DATA_PATH_SENTIMENT_LABEL = "../data/interim/manual_sentiment_labels.parquet"
df_final = load_dataframe(
    DATA_PATH_SENTIMENT_LABEL
)

df_final.shape

(20, 2)

In [5]:
# Validation
economy_df[
    [
        "date",
        "title",
        "summary"
    ]
].head()

,date,title,summary
0,2023-03-16 07:16:28,"Tok! BI Tahan Suku Bunga Acuan di Level 5,75%",Bank Indonesia memutuskan untuk mempertahankan...
1,2023-03-19 02:11:48,'Tidak Foya-foya' Tips Walkot Depok ke Warga B...,"Wali Kota Depok, Mohammad Idris, memberikan ti..."
2,2023-03-22 05:45:35,"Besok Puasa, Daging Sapi Lokal Tembus Rp140 Ri...","Harga daging sapi lokal di Pasar Timbul, Cigan..."
3,2023-03-14 09:18:39,Cara dan Syarat Pengajuan KUR BCA Online Terbaru,Bank BCA menyediakan bantuan modal berupa kred...
4,2023-03-14 11:13:41,"Depo Plumpang Belum Akan Ditutup, Ini Alasan B...",Pertamina akan melakukan perluasan buffer zone...


In [6]:
from src.data_validation import (
check_missing_values
)

missing_values = check_missing_values(
    economy_df
)

missing_values

id                 0
source             0
title              0
image              9
url                0
content            0
date               0
embedding          0
created_at         0
updated_at         0
summary            1
nomic_embedding    0
label              0
dtype: int64

In [7]:
## Create Text for Sentiment
def create_text_for_sentiment(df):
    """
    Combine title and summary
    into a single text field.

    Parameters
    ----------
    df : pandas.DataFrame
        Input dataframe.

    Returns
    -------
    pandas.DataFrame
        Dataframe with
        text_for_sentiment column.
    """

    df = df.copy()

    df["text_for_sentiment"] = (
        df["title"].fillna("") + ". " + df["summary"].fillna("")
    )

    return df

In [8]:
economy_df = create_text_for_sentiment(
    economy_df
)

In [9]:
### Validation
economy_df[
    [
        "title",
        "summary",
        "text_for_sentiment"
    ]
].sample(
    5,
    random_state=42
)

,title,summary,text_for_sentiment
1550,"BCA Syariah Cetak Laba Bersih Rp 117,6 Miliar ...",BCA Syariah mencatatkan kinerja yang positif d...,"BCA Syariah Cetak Laba Bersih Rp 117,6 Miliar ..."
1221,"SVB Kebangkrutan Bank Terbesar Sejak 2008, Mil...","Bank pemberi pinjaman startup, Silicon Valley ...","SVB Kebangkrutan Bank Terbesar Sejak 2008, Mil..."
1941,"AS Pening, 'Badai' Baru Bikin Penampakan Reses...",Presiden Fed Minneapolis Neel Kashkari menyata...,"AS Pening, 'Badai' Baru Bikin Penampakan Reses..."
2939,"Pertumbuhan Ekonomi Indonesia Ditaksir 4,9 sam...","Perry Warjiyo, Calon Gubernur BI 2023-2028, me...","Pertumbuhan Ekonomi Indonesia Ditaksir 4,9 sam..."
4970,Jokowi Buka Opsi Impor Beras 500 Ribu Ton Lagi,Presiden Joko Widodo membuka opsi impor beras ...,Jokowi Buka Opsi Impor Beras 500 Ribu Ton Lagi...


In [10]:
### Text Length Check
def calculate_text_length(df, column):
    """
    Calculate text length in words.

    Parameters
    ----------
    df : pandas.DataFrame
        Input dataframe.

    column : str
        Text column.

    Returns
    -------
    pandas.Series
        Word count.
    """
    return (
        df[column].fillna("").apply(
            lambda x: len(str(x).split())
        )
    )

In [11]:
text_length = calculate_text_length(
    economy_df,
    "text_for_sentiment"
)

text_length.describe()

count    6867.000000
mean       75.239115
std        15.321505
min         9.000000
25%        66.000000
50%        77.000000
75%        86.000000
max       131.000000
Name: text_for_sentiment, dtype: float64

In [12]:
## Save file
save_dataframe(
    economy_df,
    "../data/interim/economy_news_sentiment_input.parquet"
)

## 2. Sentiment Model Candidates

### 2.1. Candidate Models

#### Sentiment Model Candidates

In [13]:
model_candidate = {
    'Model': ['VADER', 'TextBlob', 'IndoBERT Sentiment', 'IndoBERTweet'],
    'Language': ['English', 'English', 'Indonesian', 'Indonesian'],
    'Pros': ['Fast', 'Simple', 'Strong contextual understanding', 'Good for social media'],
    'Cons': ['Poor for Indonesian', 'Poor for Indonesian', 'Heavier', 'Less suitable for news']
}

df = pd.DataFrame(model_candidate)
df

,Model,Language,Pros,Cons
0,VADER,English,Fast,Poor for Indonesian
1,TextBlob,English,Simple,Poor for Indonesian
2,IndoBERT Sentiment,Indonesian,Strong contextual understanding,Heavier
3,IndoBERTweet,Indonesian,Good for social media,Less suitable for news


### 2.2 Selected Strategy

In [14]:
selected_strategy = {
    'Aspect': [
        'Purpose',
        'Complexity',
        'Training Required',
        'Interpretability',
        'Context Understanding',
        'Computation Cost',
        'Expected Performance',
        'Reason for Selection'
    ],
    'Rule-Based Sentiment (Baseline)': [
        'Benchmark model',
        'Low',
        'No',
        'High',
        'Limited',
        'Low',
        'Moderate',
        'Fast, simple, provides comparison baseline' 
    ],
    'Transformer-Based Sentiment (Advanced)': [
        'Main sentiment model',
        'High',
        'No (Pretrained)',
        'Medium',
        'Strong',
        'Higher',
        'Better',
        'Context-aware and more suitable for economic news'
    ]
}

df = pd.DataFrame(selected_strategy)
df

,Aspect,Rule-Based Sentiment (Baseline),Transformer-Based Sentiment (Advanced)
0,Purpose,Benchmark model,Main sentiment model
1,Complexity,Low,High
2,Training Required,No,No (Pretrained)
3,Interpretability,High,Medium
4,Context Understanding,Limited,Strong
5,Computation Cost,Low,Higher
6,Expected Performance,Moderate,Better
7,Reason for Selection,"Fast, simple, provides comparison baseline",Context-aware and more suitable for economic news


### 2.3. Expected Output

In [15]:
expected_output = {
    'News': [
        'IHSG Menguat', 
        'Rupiah melemah', 
        'BI mempertahankan suku bunga'
    ],
    'Sentiment': ['Positive', 'Negative', 'Neutral'] 
}

df = pd.DataFrame(expected_output)
df

,News,Sentiment
0,IHSG Menguat,Positive
1,Rupiah melemah,Negative
2,BI mempertahankan suku bunga,Neutral


In [16]:
economy_df[
    ["title", "summary"]
].sample(
    10,
    random_state=42
)

,title,summary
1550,"BCA Syariah Cetak Laba Bersih Rp 117,6 Miliar ...",BCA Syariah mencatatkan kinerja yang positif d...
1221,"SVB Kebangkrutan Bank Terbesar Sejak 2008, Mil...","Bank pemberi pinjaman startup, Silicon Valley ..."
1941,"AS Pening, 'Badai' Baru Bikin Penampakan Reses...",Presiden Fed Minneapolis Neel Kashkari menyata...
2939,"Pertumbuhan Ekonomi Indonesia Ditaksir 4,9 sam...","Perry Warjiyo, Calon Gubernur BI 2023-2028, me..."
4970,Jokowi Buka Opsi Impor Beras 500 Ribu Ton Lagi,Presiden Joko Widodo membuka opsi impor beras ...
1511,Daftar Harga Emas Antam Hari Ini Usai Turun Rp...,Harga emas Antam turun Rp4.000 menjadi Rp1.068...
6499,"Turun Ceban, Harga Emas Antam Hari Ini Dijual ...",Harga emas Antam turun menjadi Rp1.077.000/gra...
2916,Entitas Nikel Harita Group Targetkan Pendapata...,PT Trimegah Bangun Persada Tbk menargetkan pen...
5137,"Ramai Panik Silicon Valley Bank Bangkrut, Bos ...",OJK menilai tidak ada dampak langsung yang aka...
1670,"Lakukan Transisi Energi, PGE Bukukan Pendapata...",Pertamina Geothermal Energy mendapatkan pos pe...


## 3. Sentiment Pipeline Prototype

### 3.1. Candidate Models

- Input:
  `text_for_sentiment`

- Output:

In [17]:
output = {
    'Column': ['sentiment_label', 'sentiment_score'],
    'Description': ['Positive / Neutral / Negative', '1 / 0 / -1']
}

df = pd.DataFrame(output)
df

,Column,Description
0,sentiment_label,Positive / Neutral / Negative
1,sentiment_score,1 / 0 / -1


### 3.2. Build Sentiment Functions

In [18]:
def assign_sentiment_label(text):
    """
    Assign sentiment label to a news text.

    Parameters
    ----------
    text : str
        Input news text.

    Returns
    -------
    str
        Sentiment label.
    """

In [19]:
def convert_sentiment_to_score(label):
    """
    Convert sentiment label into numerical score.

    Parameters
    ----------
    label : str
        Input sentiment label.

    Returns
    -------
    int
        Sentiment score.
    """

### 3.3. Create Small Experiment Sample

In [20]:
sample_news = economy_df['title'].sample(
    20,
    random_state=42
)

pd.set_option('display.max_colwidth', None)
sample_news

1550                               BCA Syariah Cetak Laba Bersih Rp 117,6 Miliar di 2022
1221                SVB Kebangkrutan Bank Terbesar Sejak 2008, Miliaran Dolar Tersandera
1941                          AS Pening, 'Badai' Baru Bikin Penampakan Resesi Kian Nyata
2939              Pertumbuhan Ekonomi Indonesia Ditaksir 4,9 sampai 5,7 Persen pada 2025
4970                                      Jokowi Buka Opsi Impor Beras 500 Ribu Ton Lagi
1511                            Daftar Harga Emas Antam Hari Ini Usai Turun Rp4.000/Gram
6499                      Turun Ceban, Harga Emas Antam Hari Ini Dijual Rp1.077.000/Gram
2916                  Entitas Nikel Harita Group Targetkan Pendapatan Naik 100% Usai IPO
5137                       Ramai Panik Silicon Valley Bank Bangkrut, Bos OJK Bilang Gini
1670                  Lakukan Transisi Energi, PGE Bukukan Pendapatan dari Carbon Credit
3076                        Pernah Terseret Kasus Pajak, Saham Bank Panin Masih Menarik?
3469                 

### 3.4. Manual Labeling Exercise

Widget used only for initial manual labeling.

Labels are stored in:
data/interim/manual_sentiment_labels.parquet

In [21]:
df_final

,Title,Label
0,"BCA Syariah Cetak Laba Bersih Rp 117,6 Miliar di 2022",Positive
1,"SVB Kebangkrutan Bank Terbesar Sejak 2008, Miliaran Dolar Tersandera",Negative
2,"AS Pening, 'Badai' Baru Bikin Penampakan Resesi Kian Nyata",Negative
3,"Pertumbuhan Ekonomi Indonesia Ditaksir 4,9 sampai 5,7 Persen pada 2025",Positive
4,Jokowi Buka Opsi Impor Beras 500 Ribu Ton Lagi,Neutral
5,Daftar Harga Emas Antam Hari Ini Usai Turun Rp4.000/Gram,Neutral
6,"Turun Ceban, Harga Emas Antam Hari Ini Dijual Rp1.077.000/Gram",Neutral
7,Entitas Nikel Harita Group Targetkan Pendapatan Naik 100% Usai IPO,Positive
8,"Ramai Panik Silicon Valley Bank Bangkrut, Bos OJK Bilang Gini",Negative
9,"Lakukan Transisi Energi, PGE Bukukan Pendapatan dari Carbon Credit",Positive


### 3.5. Labeling Guideline

Positive:
- Positive economic growth
- Earning growth
- Business Expansion
- Investment increase

Neutral:
- Informational news
- Policy announcement without clear positive/negative impact

Negative:
- Bankruptcy
- Recession signals
- Economic slowdown
- Financial losses

In [22]:
df_final

,Title,Label
0,"BCA Syariah Cetak Laba Bersih Rp 117,6 Miliar di 2022",Positive
1,"SVB Kebangkrutan Bank Terbesar Sejak 2008, Miliaran Dolar Tersandera",Negative
2,"AS Pening, 'Badai' Baru Bikin Penampakan Resesi Kian Nyata",Negative
3,"Pertumbuhan Ekonomi Indonesia Ditaksir 4,9 sampai 5,7 Persen pada 2025",Positive
4,Jokowi Buka Opsi Impor Beras 500 Ribu Ton Lagi,Neutral
5,Daftar Harga Emas Antam Hari Ini Usai Turun Rp4.000/Gram,Neutral
6,"Turun Ceban, Harga Emas Antam Hari Ini Dijual Rp1.077.000/Gram",Neutral
7,Entitas Nikel Harita Group Targetkan Pendapatan Naik 100% Usai IPO,Positive
8,"Ramai Panik Silicon Valley Bank Bangkrut, Bos OJK Bilang Gini",Negative
9,"Lakukan Transisi Energi, PGE Bukukan Pendapatan dari Carbon Credit",Positive


### Why Manual Labeling?

- Define economic sentiment guidelines
- Create an initial evaluation dataset
- Validate business logic
- Support future model evaluation and fine-tuning

## 4. Baseline Sentiment Model

Goal: Build a sentiment generator automatically

### 4.1. Define Baseline Approach

A rule-based sentiment approach is used as the first benchmark model.

Reasons:
- Simple and interpretable
- Fast to implement
- No training required
- Provides a baseline for future transformer-based models

### 4.2. Create Economic Sentiment Lexicon

In [23]:
def get_sentiment_keywords():
    """
    Return positive and negative
    economic sentiment keywords.
    """
    positive_words = [
     "laba",
    "untung",
    "naik",
    "menguat",
    "ekspansi",
    "investasi",
    "tumbuh",
    "buyback",
    ]

    negative_words = [
     "bangkrut",
    "resesi",
    "krisis",
    "rugi",
    "turun",
    "panic",
    ]

    return positive_words, negative_words

### 4.3. Create Rule-Based Scoring Function

In [24]:
def assign_sentiment_label(text):
    """
    Assign sentiment label to a news text.

    Parameters
    ----------
    text : str
        Input news text.

    Returns
    -------
    str
        Sentiment label.
    """
    positive_words, negative_words = get_sentiment_keywords()

    words = str(text).lower().split()

    # Positive word count
    pos_count = sum(1 for word in words if word in positive_words)
    
    # Negative word count
    neg_count = sum(1 for word in words if word in negative_words)

    # Comparing and Labeling
    if pos_count > neg_count:
        return "Positive"
    elif neg_count > pos_count:
        return "Negative"
    else:
        return "Neutral"

### 4.4. Convert Label to Score

In [25]:
def convert_sentiment_to_score(label):
    """
    Convert sentiment label into numerical score.
    """

    mapping = {
        "Positive": 1,
        "Neutral": 0,
        "Negative": -1
    }

    if label not in mapping:
        raise ValueError(
            f"Invalid sentiment label: {label}"
        )

    return mapping[label]

In [26]:
assign_sentiment_label(
    "BCA Syariah Cetak Laba dan Tumbuh Pesat"
)

'Positive'

In [27]:
assign_sentiment_label(
    "SVB bangkrut akibat krisis ekonomi"
)

'Negative'

In [28]:
assign_sentiment_label(
    "Harga emas hari ini Rp1.000.000"
)

'Neutral'

### 4.5. Test on Manual Labels

#### 4.5.1. Generate Prediction

In [29]:
## Generate Prediction
df_final["predicted_label"] = (
    df_final["Title"]
    .apply(assign_sentiment_label)
)

df_final.head()

,Title,Label,predicted_label
0,"BCA Syariah Cetak Laba Bersih Rp 117,6 Miliar di 2022",Positive,Positive
1,"SVB Kebangkrutan Bank Terbesar Sejak 2008, Miliaran Dolar Tersandera",Negative,Neutral
2,"AS Pening, 'Badai' Baru Bikin Penampakan Resesi Kian Nyata",Negative,Negative
3,"Pertumbuhan Ekonomi Indonesia Ditaksir 4,9 sampai 5,7 Persen pada 2025",Positive,Neutral
4,Jokowi Buka Opsi Impor Beras 500 Ribu Ton Lagi,Neutral,Neutral


#### 4.5.2. Check Agreement

In [30]:
df_final["match"] = (
    df_final["Label"] == df_final["predicted_label"]
)

df_final.head()

,Title,Label,predicted_label,match
0,"BCA Syariah Cetak Laba Bersih Rp 117,6 Miliar di 2022",Positive,Positive,True
1,"SVB Kebangkrutan Bank Terbesar Sejak 2008, Miliaran Dolar Tersandera",Negative,Neutral,False
2,"AS Pening, 'Badai' Baru Bikin Penampakan Resesi Kian Nyata",Negative,Negative,True
3,"Pertumbuhan Ekonomi Indonesia Ditaksir 4,9 sampai 5,7 Persen pada 2025",Positive,Neutral,False
4,Jokowi Buka Opsi Impor Beras 500 Ribu Ton Lagi,Neutral,Neutral,True


#### 4.5.3. Calculate Accuracy

In [31]:
accuracy = (
    df_final["match"]
    .mean()
)

print(
    f"Baseline agreement: {accuracy:.2%}"
)

Baseline agreement: 50.00%


#### 4.5.4. Inspect Errors

In [32]:
df_final[
    ~df_final["match"]
]

,Title,Label,predicted_label,match
1,"SVB Kebangkrutan Bank Terbesar Sejak 2008, Miliaran Dolar Tersandera",Negative,Neutral,False
3,"Pertumbuhan Ekonomi Indonesia Ditaksir 4,9 sampai 5,7 Persen pada 2025",Positive,Neutral,False
5,Daftar Harga Emas Antam Hari Ini Usai Turun Rp4.000/Gram,Neutral,Negative,False
6,"Turun Ceban, Harga Emas Antam Hari Ini Dijual Rp1.077.000/Gram",Neutral,Negative,False
8,"Ramai Panik Silicon Valley Bank Bangkrut, Bos OJK Bilang Gini",Negative,Neutral,False
9,"Lakukan Transisi Energi, PGE Bukukan Pendapatan dari Carbon Credit",Positive,Neutral,False
11,"Naik Rp 10 Ribu, Harga Emas Antam Hari Ini Rp 1.083.000 per Gram",Neutral,Positive,False
15,"Ahli: Jaga Stok Pangan, RI Bisa Produksi Beras di Luar Negeri",Positive,Neutral,False
16,"6 Cabang BNI Luar Negeri Guyur Kredit Rp 22 T, Cek Datanya",Positive,Neutral,False
19,"IHSG Diramal Menguat, Simak Rekomendasi Saham Hari Ini",Positive,Neutral,False


In [33]:
df_final["Label"].value_counts()

Label
Positive    9
Neutral     7
Negative    4
Name: count, dtype: int64

In [34]:
df_final["predicted_label"].value_counts()

predicted_label
Neutral     11
Positive     5
Negative     4
Name: count, dtype: int64

### Baseline Model Evaluation

The rule-based sentiment model achieved approximately 50% agreement with manually labeled samples.

Key observations:

- The model tends to over-predict Neutral sentiment.
- Positive news is often classified as Neutral because many economic terms are not included in the keyword dictionary.
- The model can only detect explicit keywords and cannot understand context or semantic meaning.
- Word variations such as "kebangkrutan" and "bangkrut" are treated as different tokens.
- These limitations motivate the use of transformer-based models such as IndoBERT in future experiments.

In [35]:
save_dataframe(
    df_final,
    "../data/interim/manual_sentiment_evaluation.parquet"
)